<a href="https://colab.research.google.com/github/Shanthan0/Neural-Networks-and-Deep-Learning-Coursework/blob/main/ICP_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
import re
from sklearn.preprocessing import LabelEncoder
from keras.models import load_model
import numpy as np
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

# Load and preprocess the dataset
data = pd.read_csv('/content/Sentiment (3) (2) (1).csv')
data = data[['text', 'sentiment']]

# Preprocess text data
data['text'] = data['text'].apply(lambda x: x.lower())
data['text'] = data['text'].apply(lambda x: re.sub('[^a-zA-z0-9\s]', '', x))

# Remove 'rt' from text using iloc
for idx, row in data.iterrows():
    data.iloc[idx, 0] = row['text'].replace('rt', '')

# Tokenize the text data
max_fatures = 2000
tokenizer = Tokenizer(num_words=max_fatures, split=' ')
tokenizer.fit_on_texts(data['text'].values)
X = tokenizer.texts_to_sequences(data['text'].values)
X = pad_sequences(X)

# Encode sentiment labels
labelencoder = LabelEncoder()
integer_encoded = labelencoder.fit_transform(data['sentiment'])
y = to_categorical(integer_encoded)

# Split the dataset
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Define the LSTM model
embed_dim = 128
lstm_out = 196

# Update the create_model function to include dropout_rate as a parameter
def create_model(optimizer='rmsprop',input_length=X.shape[1]):
    model = Sequential()
    model.add(Embedding(max_fatures, embed_dim, input_length=input_length))
    model.add(LSTM(lstm_out, dropout=0.2, recurrent_dropout=0.2))
    model.add(Dense(3, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# Train the model
batch_size = 32
model = create_model()
history = model.fit(X_train, Y_train, epochs=1, batch_size=batch_size, verbose=2)

# Save the model in the recommended .keras format
model.save('sentiment_lstm_model.keras')
print("Model saved successfully.")

# Evaluate the model on the test data
score, acc = model.evaluate(X_test, Y_test, verbose=2, batch_size=batch_size)
print(f"Test Score: {score}")
print(f"Test Accuracy: {acc}")

# Load the saved model for prediction
model = load_model('sentiment_lstm_model.keras')

# New text data for prediction
new_text = "A lot of good things are happening. We are respected again throughout the world, and that's a great thing.@realDonaldTrump"

# Preprocess the new text
new_text = new_text.lower()
new_text = re.sub('[^a-zA-z0-9\s]', '', new_text)
new_text = new_text.replace('rt', '')

# Tokenize and pad the new text
seq = tokenizer.texts_to_sequences([new_text])
padded = pad_sequences(seq, maxlen=X.shape[1])

# Predict the sentiment of the new text
prediction = model.predict(padded)
predicted_label = np.argmax(prediction, axis=1)
sentiment = labelencoder.inverse_transform(predicted_label)

print(f"The sentiment for the new text is: {sentiment[0]}")

estimator = KerasClassifier(build_fn=create_model, verbose=1)
# Define the grid of hyperparameters to search
param_grid = {
    'batch_size': [32, 64],
    'epochs': [3, 5],
    'optimizer': ['adam', 'rmsprop'],
}

# Initialize GridSearchCV
grid = GridSearchCV(estimator=estimator,
                    n_jobs=-1,
                    verbose=1,
                    param_grid=param_grid,)
# Fit the grid search
grid_result = grid.fit(X_train, Y_train,)

# Summarize the best results
print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")
print(f"Best Accuracy: {grid_result.best_score_}")


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


291/291 - 24s - 81ms/step - accuracy: 0.6299 - loss: 0.8584
Model saved successfully.
144/144 - 3s - 23ms/step - accuracy: 0.6579 - loss: 0.8050
Test Score: 0.8049563765525818
Test Accuracy: 0.6579292416572571
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step
The sentiment for the new text is: Negative
